In [1]:

import pandas as pd
import torch
import re
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    pipeline
)
# import evaluate
from tqdm import tqdm
import random

c:\Users\IFIX\Documents\university\مشروع تخرج\coding-articture\env-flan-t5\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============= 2. تحميل وتنظيف البيانات =============

print("📥 جاري تحميل البيانات...")
df = pd.read_csv("recipes.csv")


# الاحتفاظ بالأعمدة المهمة فقط
essential_cols = ["Name", "Calories", "ProteinContent", "RecipeIngredientParts",
                  "RecipeInstructions", "FatContent", "CarbohydrateContent"]
df = df[essential_cols].dropna()

# تحليل القوائم النصية
def parse_list(x):
    try:
        if isinstance(x, str) and x.startswith('c'):
            x = x[1:]
        # إزالة الأقواس والاقتباسات للحصول على نص نظيف
        x = x.strip('()').replace('"', '').replace("'", '')
        return x
    except:
        return str(x)

df["RecipeIngredientParts"] = df["RecipeIngredientParts"].apply(parse_list)
df["RecipeInstructions"] = df["RecipeInstructions"].apply(parse_list)

# تحويل الأنواع
df["Calories"] = pd.to_numeric(df["Calories"], errors="coerce")
df["ProteinContent"] = pd.to_numeric(df["ProteinContent"], errors="coerce")
df["FatContent"] = pd.to_numeric(df["FatContent"], errors="coerce")
df["CarbohydrateContent"] = pd.to_numeric(df["CarbohydrateContent"], errors="coerce")

# إزالة القيم غير الصالحة
df = df.dropna()


📥 جاري تحميل البيانات...


C:\Users\IFIX\AppData\Local\Temp\ipykernel_14920\114318747.py:4: DtypeWarning: Columns (0,2,14,15,16,17,18,19,20,21,22,23,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("recipes.csv")


In [3]:
# ============= 3. تنظيف النصوص =============
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # إزالة المسافات الزائدة
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # إزالة الرموز غير المرغوبة
    return text.strip()

for col in ["Name", "RecipeIngredientParts", "RecipeInstructions"]:
    df[col] = df[col].apply(clean_text)


In [4]:

# ============= 4. تحديد هدف التغذية =============
def infer_nutrition_goal(row):
    calories = row["Calories"]
    protein = row["ProteinContent"]
    carbs = row["CarbohydrateContent"]
    fat = row["FatContent"]

    if calories < 400:
        goal = "Weight Loss"
        reason = f"Low in calories ({calories:.0f} kcal)"
    elif protein >= 25:
        goal = "Muscle Building"
        reason = f"High in protein ({protein:.1f} grams)"
    elif carbs < 30 and fat < 15:
        goal = "Keto Diet"
        reason = f"Low in carbohydrates ({carbs:.1f} grams)"
    elif fat < 10:
        goal = "Low-Fat Diet"
        reason = f"Low in fat ({fat:.1f} grams)"
    else:
        goal = "Balanced Healthy Diet"
        reason = "Good balance of nutrients"

    return goal, reason


df["Goal"], df["Goal_Reason"] = zip(*df.apply(infer_nutrition_goal, axis=1))



In [5]:
len(df)

522516

In [6]:
# ============= 5.1 بناء مجموعة بيانات المحادثة =============
import random


def get_meals_by_goal(df, goal, n=5):
    """
    ترجع n وجبات عشوائية من الداتاست تناسب الهدف المطلوب
    """

    filtered = df[df["Goal"].str.lower() == str(goal).lower()].copy()

    if filtered.empty:
        return []

    n = min(n, len(filtered))

    sampled = filtered.sample(n=n, random_state=None)

    return sampled["Name"].tolist()


system_prompt = """You are a professional nutritionist specialized in providing dietary advice.
    Your tasks:
    1. Provide personalized nutrition recommendations
    2. Explain nutritional benefits clearly
    3. Design daily and weekly meal plans
    4. Suggest healthy alternatives
    Be supportive, motivating, and accurate."""

openers = [
    "Sure, let's take a look.",
    "That's a great goal to focus on.",
    "You're on the right track.",
    "Absolutely, here's a solid option for you.",
    "Good question — nutrition plays a big role here.",
]

closings = [
    "Let me know if you'd like a lighter or higher-protein option.",
    "Portion size can be adjusted based on your calorie needs.",
    "I can also create a full weekly plan if you want.",
    "Consistency matters more than perfection — keep going!",
]


def build_conversation_dataset(df, max_samples=20000):
    conversations = []

    for idx, row in tqdm(
        df.iterrows(), total=min(len(df), max_samples), desc="Building Conversations"
    ):
        if idx >= max_samples:
            break

        opener = random.choice(openers)
        closing = random.choice(closings)

        meal_list = get_meals_by_goal(df, row["Goal"], n=5)
        meal_bullets = "\n".join([f"• {meal}" for meal in meal_list])

        # ===============================
        # 1️⃣ Single meal recommendation
        # ===============================
        scenarios = [
            {
                "user": f"I need a meal for {row['Goal']}. Can you recommend something?",
                "assistant": f"""{opener}

                                **{row['Name']}** is a great option for your {row['Goal'].lower()} goal.

                                Nutrition per serving:
                                - Calories: {row['Calories']:.0f}
                                - Protein: {row['ProteinContent']:.1f}g
                                - Carbs: {row['CarbohydrateContent']:.1f}g
                                - Fat: {row['FatContent']:.1f}g

                                Why it works:
                                {row['Goal_Reason']}.

                                {closing}
                            """,
            },
            # ===============================
            # 2️⃣ Nutritional analysis
            # ===============================
            {
                "user": f"Tell me about the nutritional value of {row['Name']}",
                "assistant": f"""Here’s a nutritional breakdown of **{row['Name']}**:

                                - Energy: {row['Calories']:.0f} kcal
                                - Protein: {row['ProteinContent']:.1f}g (supports muscle and recovery)
                                - Carbohydrates: {row['CarbohydrateContent']:.1f}g (main energy source)
                                - Fat: {row['FatContent']:.1f}g

                                This meal fits well for **{row['Goal']}** because {row['Goal_Reason'].lower()}.
                            """,
            },
            # ===============================
            # 3️⃣ Preparation method
            # ===============================
            {
                "user": f"How can I prepare {row['Name']} for my {row['Goal']} diet?",
                "assistant": f"""To prepare **{row['Name']}** for your {row['Goal'].lower()} diet:

                                Ingredients:
                                {row['RecipeIngredientParts']}

                                Instructions:
                                {row['RecipeInstructions'][:500]}...

                                This preparation keeps calories controlled while preserving nutritional value.
                            """,
            },
            # ===============================
            # 4️⃣ Healthy meal list
            # ===============================
            {
                "user": f"Give me a list of healthy meals suitable for {row['Goal']}",
                "assistant": f"""Here are some healthy meal options suitable for {row['Goal'].lower()}:

                                {meal_bullets}

                                These meals were selected because they align well with your nutritional goal and support consistency in a healthy eating routine.
                                """,
            },
            # ===============================
            # 5️⃣ Daily meal plan
            # ===============================
            # {
            #     "user": f"Create a daily meal plan for {row['Goal']}",
            #     "assistant": f"""Here’s a simple daily meal plan for {row['Goal'].lower()}:
            #                     Breakfast:
            #                     Light protein-rich meal (e.g. eggs, yogurt, oats)
            #                     Lunch:
            #                     **{row['Name']}** with balanced portions
            #                     Dinner:
            #                     Vegetables + lean protein (low-calorie, easy to digest)
            #                     This structure helps maintain energy while supporting your goal.
            #                 """,
            # },
            # ===============================
            # 6️⃣ Weekly meal plan
            # ===============================
            # {
            #     "user": f"Create a 7-day meal plan for {row['Goal']} including breakfast, lunch, and dinner",
            #     "assistant": f"""Here is a sample 7-day meal plan designed for {row['Goal'].lower()}.
            #                     Day 1:
            #                     Breakfast: Light protein meal
            #                     Lunch: {row['Name']}
            #                     Dinner: Vegetables with lean protein
            #                     Day 2:
            #                     Breakfast: Oats or yogurt
            #                     Lunch: Balanced whole-food meal
            #                     Dinner: Light dinner option
            #                     Day 3–7:
            #                     Follow the same structure while rotating protein sources and vegetables.
            #                     This approach ensures balance, variety, and consistency without complexity.
            #     """,
            # },
        ]

        for scenario in scenarios:
            conversations.append(
                {
                    "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                    "target_text": scenario["assistant"],
                }
            )

    return Dataset.from_list(conversations)


print("📊 جاري بناء مجموعة البيانات...")
dataset = build_conversation_dataset(
    df,
    max_samples=2500,
)  # تحديد الحد الأقصى لعدد العينات 2000 بدلاً من 10000 للتجريب


print(f"dataset: {len(dataset)}.  ")

📊 جاري بناء مجموعة البيانات...


Building Conversations:  98%|█████████▊| 2445/2500 [08:45<00:11,  4.66it/s]


dataset: 9780.  


In [7]:
# ============= 5.2 بناء مجموعة بيانات المحادثة =============
import random
from tqdm import tqdm


def filter_meals_by_disease(df, disease, max_samples=20):

    meals = []

    for _, row in df.iterrows():

        if str(row["Disease"]).strip().lower() == str(disease).strip().lower():

            meals.append(
                {
                    "breakfast": row["Breakfast Suggestion"],
                    "lunch": row["Lunch Suggestion"],
                    "dinner": row["Dinner Suggestion"],
                }
            )

    random.shuffle(meals)

    return meals[:max_samples]


def build_daily_plan(df, disease):

    meals = filter_meals_by_disease(df, disease, max_samples=5)

    if len(meals) == 0:
        return "No meals found for this condition."

    meal = random.sample(meals, 1)[0]

    return f"""
                Here is a daily meal plan suitable for someone with {disease.lower()}:

                Breakfast:
                {meal['breakfast']}

                Lunch:
                {meal['lunch']}

                Dinner:
                {meal['dinner']}

                This plan focuses on balanced nutrition while supporting the dietary needs associated with {disease.lower()}.
            """


def build_weekly_plan(df, disease):

    meals = filter_meals_by_disease(df, disease, max_samples=20)

    if len(meals) == 0:
        return "No meals found for this condition."

    # إذا كان العدد أقل من 7، نسمح بالتكرار لإتمام خطة الأسبوع
    if len(meals) < 7:
        meals = meals * (7 // len(meals) + 1)

    plan = ""

    for i in range(7):
        meal = random.sample(meals, 1)[0]

        plan += f"""
                    Day {i+1}:

                    Breakfast:
                    {meal['breakfast']}

                    Lunch:
                    {meal['lunch']}

                    Dinner:
                    {meal['dinner']}

                """

    return f"""
                Here is a 7-day meal plan designed for people with {disease.lower()}.

                {plan}

                This weekly plan provides variety while maintaining nutritional balance suitable for managing {disease.lower()}.
            """


def build_conversation_dataset_detailed_meals_macros(df, max_samples=20000):
    conversation = []

    for idx, row in tqdm(
        df.iterrows(),
        total=min(len(df), max_samples),
        desc="Building Detailed Meal Conversations",
    ):
        if idx >= max_samples:
            break

        opener = random.choice(openers)
        closing = random.choice(closings)

        scenarios = [
            # تحليل يومي مفصل
            {
                "user": f"Can you create a daily meal plan for a {row['Activity Level'].lower()} {row['Gender'].lower()} aiming for {row['Dietary Preference'].lower()} diet?",
                "assistant": f"{opener}\n\nHere's a daily meal plan based on your profile:\n\n**Breakfast:** {row['Breakfast Suggestion']} ({row['Breakfast Calories']} kcal, {row['Breakfast Protein']}g protein, {row['Breakfast Carbohydrates']}g carbs, {row['Breakfast Fats']}g fat)\n\n**Lunch:** {row['Lunch Suggestion']} ({row['Lunch Calories']} kcal, {row['Lunch Protein']}g protein, {row['Lunch Carbohydrates']}g carbs, {row['Lunch Fats']}g fat)\n\n**Dinner:** {row['Dinner Suggestion']} ({row['Dinner Calories']} kcal, {row['Dinner Protein.1']}g protein, {row['Dinner Carbohydrates.1']}g carbs, {row['Dinner Fats']}g fat)\n\n**Snack:** {row['Snack Suggestion']} ({row['Snacks Calories']} kcal, {row['Snacks Protein']}g protein, {row['Snacks Carbohydrates']}g carbs, {row['Snacks Fats']}g fat)\n\n{closing}",
            },
            # تحليل وجبة الفطار
            {
                "user": f"What is the nutritional breakdown for {row['Breakfast Suggestion']}?",
                "assistant": f"{opener}\n\n**{row['Breakfast Suggestion']}** provides:\n- Calories: {row['Breakfast Calories']} kcal\n- Protein: {row['Breakfast Protein']} g\n- Carbs: {row['Breakfast Carbohydrates']} g\n- Fat: {row['Breakfast Fats']} g\n\nIt is a good choice for a {row['Dietary Preference'].lower()} diet aiming for {row['Daily Calorie Target']} kcal.\n\n{closing}",
            },
            # تحليل وجبة الغداء
            {
                "user": f"Suggest a healthy lunch for a {row['Dietary Preference'].lower()} diet",
                "assistant": f"{opener}\n\nA healthy lunch option is **{row['Lunch Suggestion']}** with {row['Lunch Protein']}g protein, {row['Lunch Carbohydrates']}g carbs, and {row['Lunch Fats']}g fat.\n\nIt aligns well with a {row['Daily Calorie Target']} kcal daily target.\n\n{closing}",
            },
            # تحليل وجبة العشاء
            {
                "user": f"What should I eat for dinner to meet my protein goal?",
                "assistant": f"{opener}\n\n**{row['Dinner Suggestion']}** is a great dinner option:\n- Calories: {row['Dinner Calories']} kcal\n- Protein: {row['Dinner Protein.1']} g\n- Carbs: {row['Dinner Carbohydrates.1']} g\n- Fat: {row['Dinner Fats']} g\n\nIt helps you reach your protein requirements for the day.\n\n{closing}",
            },
            # برنامج غذائي يومي مفصل
            {
                "user": f"Create a daily meal plan for someone with {row['Disease']}",
                "assistant": build_daily_plan(df, row["Disease"]),
            },
            # برنامج غذائي أسبوعي مفصل
            {
                "user": f"Create a 7-day meal plan for someone with {row['Disease']}",
                "assistant": build_weekly_plan(df, row["Disease"]),
            },
        ]

        for scenario in scenarios:
            conversation.append(
                {
                    "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                    "target_text": scenario["assistant"],
                }
            )

    return Dataset.from_list(conversation)

# merging datasets
# from datasets import concatenate_datasets
detailed_meals_macros = pd.read_csv("detailed_meals_macros_CLEANED.csv")

# detailed_meals_macros.head()


dataset_detailed_meals_macros = build_conversation_dataset_detailed_meals_macros(
    detailed_meals_macros, max_samples=1000000
)  # تحديد الحد الأقصى لعدد العينات 2000 بدلاً من


# dataset = concatenate_datasets([dataset, dataset_detailed_meals_macros])

Building Detailed Meal Conversations: 100%|██████████| 1698/1698 [04:13<00:00,  6.69it/s]


In [9]:
dataset_detailed_meals_macros

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 10188
})

هون أنا أنشأت وجبات مشهورة في الشرق الأوسط ورح  أبني محادثات جديدة باستخدانها 

In [8]:
arab_meals = pd.read_csv("arab_meals_120.csv")

df.columns = df.columns.str.strip()


# System prompt
system_prompt = """
    You are a professional nutritionist specialized in sports nutrition and healthy meal planning.
    Your tasks:
        1. Recommend suitable meals based on the user's goal
        2. Explain nutritional values clearly
        3. Compare meals intelligently
        4. Suggest healthy substitutions
        5. Refine meal suggestions based on follow-up requests
    Be practical, supportive, and concise.
"""


openers = [
    "Sure, here’s a good option for you.",
    "Absolutely — this is a solid choice.",
    "That’s a practical and nutritious option.",
    "Yes, here’s something that fits well.",
    "Good choice — let’s break it down."
]

closings = [
    "Let me know if you'd like a cheaper or higher-protein option.",
    "I can also suggest a lower-calorie alternative if you want.",
    "If you'd like, I can compare it with another Arabic meal.",
    "I can also make it more suitable for post-workout recovery.",
    "Let me know if you want a breakfast, lunch, or dinner alternative."
]

#  Helper functions

def get_similar_meals(df, goal=None, meal_type=None, exclude_name=None):
    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower() ]

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower() ]

    if exclude_name:
        filtered = filtered[filtered["Name"].str.lower() != exclude_name.lower() ]

    return filtered.sample(min(len(filtered), 10)) if len(filtered) > 0 else filtered



def choose_substitute(row, df):
    substitutes = get_similar_meals(
        df,
        goal=row["Goal"],
        meal_type=row["Meal Type"],
        exclude_name=row["Name"]
    )

    if len(substitutes) == 0:
        return None

    return substitutes.sample(1).iloc[0]


def choose_comparison_meal(row, df):
    comparison_df = df[
        (df["Meal Type"] == row["Meal Type"]) &
        (df["Name"] != row["Name"])
    ]

    if len(comparison_df) == 0:
        return None

    return comparison_df.sample(1).iloc[0]



def compare_meals_text(row1, row2):
    better_for_goal = row1 if row1["ProteinContent"] >= row2["ProteinContent"] else row2

    comparison = f"""Here’s a comparison between **{row1['Name']}** and **{row2['Name']}**:

                    **{row1['Name']}**
                    - Calories: {row1['Calories']}
                    - Protein: {row1['ProteinContent']}g
                    - Carbs: {row1['CarbohydrateContent']}g
                    - Fat: {row1['FatContent']}g

                    **{row2['Name']}**
                    - Calories: {row2['Calories']}
                    - Protein: {row2['ProteinContent']}g
                    - Carbs: {row2['CarbohydrateContent']}g
                    - Fat: {row2['FatContent']}g

                    For **{row1['Goal'].lower()}**, **{better_for_goal['Name']}** is generally the better option because it provides stronger protein support."""

    return comparison




def build_followup_response(row, followup_type, df):
    if followup_type == "higher protein":
        alternatives = df[
            (df["Meal Type"] == row["Meal Type"]) &
            (df["ProteinContent"] > row["ProteinContent"])
        ]
        if len(alternatives) > 0:
            alt = alternatives.sample(1).iloc[0]
            return f"""If you want something **higher in protein**, a better option would be **{alt['Name']}**.

                        It provides:
                        - Calories: {alt['Calories']}
                        - Protein: {alt['ProteinContent']}g
                        - Carbs: {alt['CarbohydrateContent']}g
                        - Fat: {alt['FatContent']}g

                        This makes it more suitable if your priority is protein intake.
                    """
        else:
            return "This meal is already one of the higher-protein options in this category."

    elif followup_type == "lower calories":
        alternatives = df[
            (df["Meal Type"] == row["Meal Type"]) &
            (df["Calories"] < row["Calories"])
        ]
        if len(alternatives) > 0:
            alt = alternatives.sample(1).iloc[0]
            return f"""If you want something **lower in calories**, try **{alt['Name']}** instead.

                        It provides:
                        - Calories: {alt['Calories']}
                        - Protein: {alt['ProteinContent']}g
                        - Carbs: {alt['CarbohydrateContent']}g
                        - Fat: {alt['FatContent']}g

                        It’s a lighter option while still keeping decent nutritional value.
                    """
        else:
            return "This meal is already among the lighter options in this category."

    elif followup_type == "less fat":
        alternatives = df[
            (df["Meal Type"] == row["Meal Type"]) &
            (df["FatContent"] < row["FatContent"])
        ]
        if len(alternatives) > 0:
            alt = alternatives.sample(1).iloc[0]
            return f"""If you want something **lower in fat**, a better option would be **{alt['Name']}**.

                        It provides:
                        - Calories: {alt['Calories']}
                        - Protein: {alt['ProteinContent']}g
                        - Carbs: {alt['CarbohydrateContent']}g
                        - Fat: {alt['FatContent']}g

                        That makes it easier to fit into a lower-fat eating plan.
                    """
        else:
            return "This meal is already relatively low in fat."

    return "I can help refine it further if you want."






# Main coversation builder
def build_advanced_conversation_dataset(df, max_samples=120):
    conversations = []

    for idx, row in tqdm(
        df.iterrows(),
        total=min(len(df), max_samples),
        desc="Building Advanced Meal Conversations"
    ):
        if idx >= max_samples:
            break

        opener = random.choice(openers)
        closing = random.choice(closings)

        meal_name = row["Name"]
        goal = row["Goal"]
        meal_type = row["Meal Type"]


        # =========================
        # 1) Single meal recommendation
        # =========================
        scenarios = [
            {
                "user": f"Suggest an {meal_type.lower()} for {goal.lower()}",
                "assistant": f"""{opener}

                                **{meal_name}** is a strong option for **{goal.lower()}**.

                                **Meal type:** {meal_type}
                                **Calories:** {row['Calories']}
                                **Protein:** {row['ProteinContent']}g
                                **Carbs:** {row['CarbohydrateContent']}g
                                **Fat:** {row['FatContent']}g

                                **Why it works:**
                                It fits well into a {goal.lower()} nutrition plan and is a familiar {row['Cuisine']} choice.

                                **Main ingredients:**
                                {row['Ingredients']}

                                {closing}
                            """
            },

            # =========================
            # 2) Nutritional breakdown
            # =========================
            {
                "user": f"What are the nutritional values of {meal_name}?",
                "assistant": f"""{opener}

                                Here’s the nutritional breakdown for **{meal_name}**:

                                - **Calories:** {row['Calories']} kcal
                                - **Protein:** {row['ProteinContent']} g
                                - **Carbohydrates:** {row['CarbohydrateContent']} g
                                - **Fat:** {row['FatContent']} g

                                This meal is generally suitable for **{goal.lower()}** and works well as a **{meal_type.lower()}** option.

                                {closing}
                            """
            }
        ]

        # =========================
        # 3) Meal comparison
        # =========================
        comparison_meal = choose_comparison_meal(row, df)
        if comparison_meal is not None:
            scenarios.append({
                "user": f"Which is better for {goal.lower()}: {meal_name} or {comparison_meal['Name']}?",
                "assistant": compare_meals_text(row, comparison_meal)
            })

        # =========================
        # 4) Meal substitution
        # =========================
        substitute = choose_substitute(row, df)
        if substitute is not None:
            scenarios.append({
                "user": f"What can I eat instead of {meal_name} for {goal.lower()}?",
                "assistant": f"""{opener}

                                A good alternative to **{meal_name}** would be **{substitute['Name']}**.

                                Here’s why:
                                - It is also suitable for **{goal.lower()}**
                                - It works as a **{substitute['Meal Type'].lower()}**
                                - It provides:
                                - Calories: {substitute['Calories']}
                                - Protein: {substitute['ProteinContent']}g
                                - Carbs: {substitute['CarbohydrateContent']}g
                                - Fat: {substitute['FatContent']}g

                                **Ingredients:**
                                {substitute['Ingredients']}

                                {closing}
                            """
            })

        # =========================
        # 5) Follow-up conversational refinement
        # =========================
        followup_options = ["higher protein", "lower calories", "less fat"]

        for followup_type in followup_options:
            scenarios.append({
                "user": f"Suggest an {meal_type.lower()} for {goal.lower()}",
                "assistant": f"""{opener}

                                A good option is **{meal_name}**.

                                - Calories: {row['Calories']}
                                - Protein: {row['ProteinContent']}g
                                - Carbs: {row['CarbohydrateContent']}g
                                - Fat: {row['FatContent']}g

                                It’s a practical choice for **{goal.lower()}**.

                                {closing}
                            """,
                "followup_user": f"I want something {followup_type}",
                "followup_assistant": build_followup_response(row, followup_type, df)
            })

        # =========================
        # Save scenarios
        # =========================
        for scenario in scenarios:
            if "followup_user" in scenario:
                conversations.append({
                    "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>\n<assistant>{scenario['assistant']}</assistant>\n<user>{scenario['followup_user']}</user>",
                    "target_text": scenario["followup_assistant"]
                })
            else:
                conversations.append({
                    "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                    "target_text": scenario["assistant"]
                })

    return Dataset.from_list(conversations)


arab_meals_dataset = build_advanced_conversation_dataset(arab_meals, max_samples=1000000)

arab_meals_dataset

Building Advanced Meal Conversations: 100%|██████████| 349/349 [00:01<00:00, 224.85it/s]


Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 2443
})

In [9]:
# =========================
# 2) System prompt
# =========================
system_prompt = """You are a professional nutritionist specialized in sports nutrition and practical meal planning.
                    Your tasks:
                    1. Explain nutrition concepts clearly
                    2. Suggest healthier food alternatives
                    3. Guide users on smart restaurant and fast food choices
                    4. Recommend budget-friendly and quick meal ideas
                    Be practical, supportive, accurate, and easy to understand.
                """

# =========================
# 3) Openers / Closings
# =========================
openers = [
    "Sure — here’s a helpful explanation.",
    "Absolutely, let’s keep it practical.",
    "Good question — this matters a lot in nutrition.",
    "Yes, here’s a smart way to think about it.",
    "Here’s a simple and useful answer.",
]

closings = [
    "I can also suggest meals based on this idea if you want.",
    "Let me know if you'd like Arabic meal examples too.",
    "I can turn this into a daily eating plan if you'd like.",
    "If you want, I can also give you gym-friendly versions.",
    "I can also make this more budget-friendly if needed.",
]


def get_budget_meals(df, goal=None, meal_type=None, top_n=3):
    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    filtered = filtered.copy()
    filtered["ingredient_count"] = filtered["Ingredients"].apply(
        lambda x: len(str(x).split(","))
    )
    filtered = filtered.sort_values(
        by=["ingredient_count", "Calories"], ascending=[True, True]
    )

    return filtered.head(top_n)


def get_quick_meals(df, goal=None, meal_type=None, top_n=3):
    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    filtered = filtered.copy()
    # تقريبياً: الوجبات ذات المكونات الأقل = أسرع
    filtered["ingredient_count"] = filtered["Ingredients"].apply(
        lambda x: len(str(x).split(","))
    )
    filtered = filtered.sort_values(by=["ingredient_count"], ascending=True)

    return filtered.head(top_n)


# =========================
# Helper functions
# =========================
def safe_sample(dataframe, n=1):
    if len(dataframe) == 0:
        return None
    return dataframe.sample(min(n, len(dataframe)))


def get_meals(df, meal_type=None, goal=None):
    filtered = df.copy()

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    return filtered


def get_pre_workout_meals(df, goal=None):
    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    # تقريب منطقي:
    # قبل التمرين نفضل كارب متوسط + دهون أقل نسبيًا
    filtered = filtered[
        (filtered["CarbohydrateContent"] >= 20) & (filtered["FatContent"] <= 20)
    ]

    return filtered


def get_post_workout_meals(df, goal=None):
    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    # تقريب منطقي:
    # بعد التمرين نفضل بروتين جيد + كارب جيد
    filtered = filtered[
        (filtered["ProteinContent"] >= 15) & (filtered["CarbohydrateContent"] >= 20)
    ]

    return filtered


def get_high_protein_meals(df, meal_type=None):
    filtered = df.copy()

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    return filtered.sort_values(by="ProteinContent", ascending=False)


def get_lighter_meals(df, meal_type=None):
    filtered = df.copy()

    if meal_type:
        filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    return filtered.sort_values(by="Calories", ascending=True)


def explain_pre_workout(row):
    reasons = []

    if row["CarbohydrateContent"] >= 25:
        reasons.append("it provides useful carbohydrates for training energy")
    if row["ProteinContent"] >= 15:
        reasons.append("it also gives a decent amount of protein")
    if row["FatContent"] <= 15:
        reasons.append(
            "it is not too heavy in fat, which may make digestion easier before training"
        )

    if not reasons:
        reasons.append("it is a practical meal before training")

    return " because " + ", ".join(reasons) + "."


def explain_post_workout(row):
    reasons = []

    if row["ProteinContent"] >= 20:
        reasons.append("it supports muscle recovery with a solid amount of protein")
    if row["CarbohydrateContent"] >= 25:
        reasons.append("it helps replenish energy stores after training")
    if row["Calories"] <= 700:
        reasons.append("it is also practical to fit into a structured nutrition plan")

    if not reasons:
        reasons.append("it is a practical recovery meal")

    return " because " + ", ".join(reasons) + "."


def followup_refinement_response(row, followup_type, df, goal=None):
    meal_type = row["Meal Type"]

    filtered = df.copy()

    if goal:
        filtered = filtered[filtered["Goal"].str.lower() == goal.lower()]

    filtered = filtered[filtered["Meal Type"].str.lower() == meal_type.lower()]

    if followup_type == "lighter":
        lighter = filtered.sort_values(by="Calories", ascending=True)
        lighter = lighter[lighter["Calories"] < row["Calories"]]

        if len(lighter) > 0:
            alt = lighter.iloc[0]
            return f"""If you want something **lighter**, a better option would be **{alt['Name']}**.

- Calories: {alt['Calories']}
- Protein: {alt['ProteinContent']}g
- Carbs: {alt['CarbohydrateContent']}g
- Fat: {alt['FatContent']}g

It is easier to digest and lower in total calories."""
        else:
            return "This meal is already one of the lighter options in this category."

    elif followup_type == "higher protein":
        hp = filtered.sort_values(by="ProteinContent", ascending=False)
        hp = hp[hp["ProteinContent"] > row["ProteinContent"]]

        if len(hp) > 0:
            alt = hp.iloc[0]
            return f"""If you want something **higher in protein**, try **{alt['Name']}** instead.

- Calories: {alt['Calories']}
- Protein: {alt['ProteinContent']}g
- Carbs: {alt['CarbohydrateContent']}g
- Fat: {alt['FatContent']}g

This can be a stronger option if protein intake is your priority."""
        else:
            return "This meal is already one of the higher-protein options."

    elif followup_type == "faster":
        return f"""If you want something **faster**, you can simplify **{row['Name']}** by reducing prep and using ready ingredients.

For example:
- Use pre-cooked protein if available
- Keep the carb source simple
- Avoid complex sides

That keeps it more practical on busy training days."""

    elif followup_type == "no cooking":
        return f"""If you want a **no-cook alternative**, try a simple meal with:
- yogurt or laban
- fruit such as banana or dates
- bread or oats
- a protein source like labneh, cheese, or ready cooked chicken

This keeps it practical while still supporting training."""

    return "I can refine it further based on your goal and workout timing."


nutrition_education_topics = pd.read_csv("nutrition_education_topics.csv").to_dict(orient="records")
healthy_alternatives = pd.read_csv("healthy_alternatives.csv").to_dict(orient="records")
restaurant_guidance = pd.read_csv("restaurant_guidance.csv").to_dict(orient="records")
sports_education = pd.read_csv("sports_education.csv").to_dict(orient="records")

# =========================
# 8) Main builder
# =========================
def build_education_alternatives_restaurant_budget_dataset(df):
    conversations = []

    # -----------------------------------------
    # A) Nutrition education / explanation
    # -----------------------------------------
    for item in nutrition_education_topics:
        opener = random.choice(openers)
        closing = random.choice(closings)

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>{item['question']}</user>",
                "target_text": f"""{opener}

                                {item['answer']}

                                {closing}
                            """,
            }
        )

    # -----------------------------------------
    # B) Healthy alternatives
    # -----------------------------------------
    for item in healthy_alternatives:
        opener = random.choice(openers)
        closing = random.choice(closings)

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>What is a healthier alternative to {item['unhealthy']}?</user>",
                "target_text": f"""{opener}

                                A healthier alternative to **{item['unhealthy']}** would be **{item['healthy']}**.

                                **Why it’s better:**
                                {item['reason']}

                                This makes it easier to improve food quality without making your meals feel too restrictive.

                                {closing}
                            """,
            }
        )

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>Can you replace {item['unhealthy']} with something healthier?</user>",
                "target_text": f"""{opener}

                                Yes — a better option would be **{item['healthy']}**.

                                It works well because:
                                - It is usually more balanced
                                - It often provides better satiety
                                - It can fit more easily into a healthy eating plan

                                **Reason:**
                                {item['reason']}

                                {closing}
                            """,
            }
        )

    # -----------------------------------------
    # C) Restaurant / fast food guidance
    # -----------------------------------------
    for item in restaurant_guidance:
        opener = random.choice(openers)
        closing = random.choice(closings)

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>What should I order from a {item['restaurant_type']} for {item['goal']}?</user>",
                "target_text": f"""{opener}

                                If your goal is **{item['goal']}**, a smart choice at a **{item['restaurant_type']}** would be:

                                {item['recommendation']}

                                Try to focus on protein first, then choose carbs and sauces based on your goal.

                                {closing}
                            """,
            }
        )

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>What is the healthiest option at a {item['restaurant_type']}?</user>",
                "target_text": f"""{opener}

                                One of the healthier options at a **{item['restaurant_type']}** is usually a meal based on **grilled protein**, moderate carbs, and fewer heavy sauces.

                                For example:
                                {item['recommendation']}

                                This approach helps keep the meal more balanced and easier to fit into your nutrition plan.

                                {closing}
                            """,
            }
        )

    # -----------------------------------------
    # D) Budget / quick meal ideas from dataset
    # -----------------------------------------
    goals = df["Goal"].dropna().unique().tolist()
    meal_types = df["Meal Type"].dropna().unique().tolist()

    for goal in goals:
        for meal_type in meal_types:
            opener = random.choice(openers)
            closing = random.choice(closings)

            # Budget meals
            budget_meals = get_budget_meals(df, goal=goal, meal_type=meal_type, top_n=3)

            if len(budget_meals) > 0:
                meal_lines = []
                for _, row in budget_meals.iterrows():
                    meal_lines.append(
                        f"- **{row['Name']}** ({row['Calories']} kcal, {row['ProteinContent']}g protein)"
                    )

                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>Give me cheap {meal_type.lower()} ideas for {goal.lower()}</user>",
                        "target_text": f"""{opener}

                                        Here are some **budget-friendly {meal_type.lower()} ideas** for **{goal.lower()}**:

                                        {chr(10).join(meal_lines)}

                                        These meals are relatively simple, practical, and easier to repeat consistently.

                                        {closing}
                                    """,
                    }
                )

            # Quick meals
            quick_meals = get_quick_meals(df, goal=goal, meal_type=meal_type, top_n=3)

            if len(quick_meals) > 0:
                meal_lines = []
                for _, row in quick_meals.iterrows():
                    meal_lines.append(
                        f"- **{row['Name']}** ({row['Calories']} kcal, {row['ProteinContent']}g protein)"
                    )

                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>Suggest quick {meal_type.lower()} meals for {goal.lower()}</user>",
                        "target_text": f"""{opener}

                                        Here are some **quick {meal_type.lower()} options** for **{goal.lower()}**:

                                        {chr(10).join(meal_lines)}

                                        These meals are useful when you want something simple without spending too much time cooking.

                                        {closing}
                                    """,
                    }
                )

                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>I don't have time to cook. What are some fast {meal_type.lower()} meals for {goal.lower()}?</user>",
                        "target_text": f"""{opener}

                                        If you’re short on time, these **fast {meal_type.lower()} ideas** can work well for **{goal.lower()}**:

                                        {chr(10).join(meal_lines)}

                                        They’re practical, familiar, and easier to prepare on busy days.

                                        {closing}
                                    """,
                    }
                )

    # -------------------------
    # Knowledge / education
    # -------------------------
    for item in sports_education:
        opener = random.choice(openers)
        closing = random.choice(closings)

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>{item['question']}</user>",
                "target_text": f"""{opener}

                                {item['answer']}

                                {closing}
                            """,
            }
        )

    # -------------------------
    # B) Pre-workout meals
    # -------------------------
    for goal in goals:
        pre_df = get_pre_workout_meals(df, goal=goal)

        for _, row in pre_df.head(15).iterrows():
            opener = random.choice(openers)
            closing = random.choice(closings)

            scenarios = [
                {
                    "user": f"Suggest a pre workout meal for {goal.lower()}",
                    "assistant": f"""{opener}

                                    A good **pre-workout option** for **{goal.lower()}** is **{row['Name']}**.

                                    - Meal type: {row['Meal Type']}
                                    - Calories: {row['Calories']}
                                    - Protein: {row['ProteinContent']}g
                                    - Carbs: {row['CarbohydrateContent']}g
                                    - Fat: {row['FatContent']}g

                                    This works well before training{explain_pre_workout(row)}

                                    {closing}
                                """,
                },
                {
                    "user": f"What should I eat 1 hour before the gym for {goal.lower()}?",
                    "assistant": f"""{opener}

                                        About **1 hour before the gym**, **{row['Name']}** can be a practical option.

                                        It provides:
                                        - {row['Calories']} kcal
                                        - {row['ProteinContent']}g protein
                                        - {row['CarbohydrateContent']}g carbs
                                        - {row['FatContent']}g fat

                                        This can help support your workout without being unnecessarily heavy.

                                        {closing}
                                    """,
                },
                {
                    "user": f"Is {row['Name']} good before a workout?",
                    "assistant": f"""{opener}

                                        Yes — **{row['Name']}** can work **before a workout**{explain_pre_workout(row)}

                                        **Nutrition:**
                                        - Calories: {row['Calories']}
                                        - Protein: {row['ProteinContent']}g
                                        - Carbs: {row['CarbohydrateContent']}g
                                        - Fat: {row['FatContent']}g

                                        It can be a solid pre-training choice depending on portion size and timing.

                                        {closing}
                                """,
                },
            ]

            for scenario in scenarios:
                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                        "target_text": scenario["assistant"],
                    }
                )

            # Follow-up
            for followup_type in ["lighter", "higher protein", "faster", "no cooking"]:
                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>Suggest a pre workout meal for {goal.lower()}</user>\n<assistant>{scenarios[0]['assistant']}</assistant>\n<user>I want something {followup_type}</user>",
                        "target_text": followup_refinement_response(
                            row, followup_type, df, goal
                        ),
                    }
                )

    # -------------------------
    # C) Post-workout meals
    # -------------------------
    for goal in goals:
        post_df = get_post_workout_meals(df, goal=goal)

        for _, row in post_df.head(15).iterrows():
            opener = random.choice(openers)
            closing = random.choice(closings)

            scenarios = [
                {
                    "user": f"Suggest a post workout meal for {goal.lower()}",
                    "assistant": f"""{opener}

                                        A good **post-workout meal** for **{goal.lower()}** is **{row['Name']}**.

                                        - Meal type: {row['Meal Type']}
                                        - Calories: {row['Calories']}
                                        - Protein: {row['ProteinContent']}g
                                        - Carbs: {row['CarbohydrateContent']}g
                                        - Fat: {row['FatContent']}g

                                        This works well after training{explain_post_workout(row)}

                                        {closing}
                                    """,
                },
                {
                    "user": f"What should I eat after the gym for {goal.lower()}?",
                    "assistant": f"""{opener}

                                    After the gym, **{row['Name']}** is a practical recovery meal.

                                    It provides:
                                    - {row['Calories']} kcal
                                    - {row['ProteinContent']}g protein
                                    - {row['CarbohydrateContent']}g carbs
                                    - {row['FatContent']}g fat

                                    This makes it a solid option for post-training recovery and energy replenishment.

                                    {closing}
                                """,
                },
                {
                    "user": f"Is {row['Name']} good after a workout?",
                    "assistant": f"""{opener}

                                    Yes — **{row['Name']}** can be a good **post-workout meal**{explain_post_workout(row)}

                                    **Nutrition:**
                                    - Calories: {row['Calories']}
                                    - Protein: {row['ProteinContent']}g
                                    - Carbs: {row['CarbohydrateContent']}g
                                    - Fat: {row['FatContent']}g

                                    It can fit well into a gym-focused meal plan.

                                    {closing}
                                """,
                },
            ]

            for scenario in scenarios:
                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                        "target_text": scenario["assistant"],
                    }
                )

            # Follow-up
            for followup_type in ["lighter", "higher protein", "faster", "no cooking"]:
                conversations.append(
                    {
                        "input_text": f"<system>{system_prompt}</system>\n<user>Suggest a post workout meal for {goal.lower()}</user>\n<assistant>{scenarios[0]['assistant']}</assistant>\n<user>I want something {followup_type}</user>",
                        "target_text": followup_refinement_response(
                            row, followup_type, df, goal=goal
                        ),
                    }
                )

    # -------------------------
    # D) Recovery meal scenarios
    # -------------------------
    recovery_df = df[(df["ProteinContent"] >= 18) & (df["CarbohydrateContent"] >= 20)]

    for _, row in recovery_df.head(20).iterrows():
        opener = random.choice(openers)
        closing = random.choice(closings)

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>What is a good recovery meal after training?</user>",
                "target_text": f"""{opener}

                                A good recovery meal is **{row['Name']}**.

                                It provides:
                                - Calories: {row['Calories']}
                                - Protein: {row['ProteinContent']}g
                                - Carbs: {row['CarbohydrateContent']}g
                                - Fat: {row['FatContent']}g

                                It supports recovery because it helps provide protein for muscle repair and carbohydrates for restoring energy.

                                {closing}
                            """,
            }
        )

        conversations.append(
            {
                "input_text": f"<system>{system_prompt}</system>\n<user>What should I eat for muscle recovery?</user>",
                "target_text": f"""{opener}

                                For **muscle recovery**, **{row['Name']}** is a strong option.

                                Why it helps:
                                - Good protein support
                                - Useful carbohydrates after training
                                - Practical as a real meal, not just a snack

                                **Nutrition:**
                                - Calories: {row['Calories']}
                                - Protein: {row['ProteinContent']}g
                                - Carbs: {row['CarbohydrateContent']}g
                                - Fat: {row['FatContent']}g

                                {closing}
                            """,
            }
        )

    return Dataset.from_list(conversations)


# =========================
# 9) Build dataset
# =========================
dataset_extra = build_education_alternatives_restaurant_budget_dataset(arab_meals)

print(dataset_extra)
print("Total conversations:", len(dataset_extra))
print(dataset_extra[0])

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 1540
})
Total conversations: 1540
{'input_text': '<system>You are a professional nutritionist specialized in sports nutrition and practical meal planning.\n                    Your tasks:\n                    1. Explain nutrition concepts clearly\n                    2. Suggest healthier food alternatives\n                    3. Guide users on smart restaurant and fast food choices\n                    4. Recommend budget-friendly and quick meal ideas\n                    Be practical, supportive, accurate, and easy to understand.\n                </system>\n<user>Why is protein important after a workout?</user>', 'target_text': 'Here’s a simple and useful answer.\n\n                                Protein is important after a workout because it helps repair muscle tissue and supports muscle growth. After training, your muscles need amino acids to recover efficiently. A meal or snack with good-quality protein can impr

In [10]:
from datasets import concatenate_datasets

dataset = concatenate_datasets([dataset, dataset_detailed_meals_macros, arab_meals_dataset, dataset_extra ])



print( len(dataset) )

23951


لحد الآن تم بناء محادثات:

1️⃣ Single meal recommendation

2️⃣ Nutritional analysis

3️⃣ Preparation method

4️⃣ Healthy meal list

5️⃣ برنامج غذائي يومي مفصل - برنامج غذائي أسبوعي مفصل

6️⃣  تحليل وجبة الفطار - تحليل وجبة الغداء - تحليل وجبة العشاء

7️⃣ Meal comparison -  Meal substitution 

8️⃣ Follow-up conversational refinement

9️⃣ Knowledge base for nutrition education 

🔟 Healthy alternatives knowledge

1️⃣1️⃣ Restaurant / fast food guidance knowledge

1️⃣2️⃣ Budget / quick meal helpers

1️⃣3️⃣  Quick meals 

1️⃣4️⃣ وجبات قبل التمرين - وجبات بعد التمرين - تقريب منطقي كارب وبروتين حبوب

1️⃣5️⃣ Healthy Alternatives

Restaurant / Fast Food Guidance

Restaurant / Fast Food Guidance

Sports Nutrition / Gym Nutrition

Recovery meals

In [15]:
# تحويل Dataset إلى DataFrame ثم حفظها كـ CSV
conversations_df = dataset.to_pandas()
conversations_df.to_csv("conversations.csv", index=False, encoding="utf-8-sig")

print(f"✅ تم حفظ {len(conversations_df)} محادثة في ملف conversations.csv")

✅ تم حفظ 69955 محادثة في ملف conversations.csv
